# PIES Files

In [5]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

In [3]:
df_map=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\01_DRA\00_Projects\99_Automation_Projects\89_Autocare_Extract\Output\20260129_Autocare_PCAdb.xlsx")
df_map.PAID=df_map.PAID.astype(str)
df_map.PartTerminologyID=df_map.PartTerminologyID.astype(str)
df_map

,CategoryID,CategoryName,PartTerminologyID,PartTerminologyName,PAID,PAName
0,1,Accessories,1020,Car Cover,14,Length
1,1,Accessories,1020,Car Cover,24,Width
2,1,Accessories,1020,Car Cover,10,Material
3,1,Accessories,1020,Car Cover,17,Color
4,1,Accessories,1020,Car Cover,513,Vented
...,...,...,...,...,...,...
142133,46,"Oil, Fluids and Chemicals",68086,Intercooler Fluid,3145,Universal Or Specific Fit
142134,46,"Oil, Fluids and Chemicals",68086,Intercooler Fluid,3390,Grade Type
142135,46,"Oil, Fluids and Chemicals",70868,Wheel Tape,14,Length
142136,46,"Oil, Fluids and Chemicals",70868,Wheel Tape,24,Width


In [14]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file (update the filename if needed)
xml_file = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\OReilly\Input\Updates\PIES_FULL_GQXS_Murray Temperature Control_20260219_190932.xml"
tree = ET.parse(xml_file)
root = tree.getroot()

tree = ET.parse(xml_file)
root = tree.getroot()

# Define namespace if present (adjust if needed)
ns = {"ns": "http://www.autocare.org"}

# Find all Item elements
items = root.findall(".//ns:Item", ns)

extended_info_data = []
product_attr_data = []

# Loop through each Item element and extract the desired values.
for item in items:
    # Get the PartNumber and PartTerminologyID
    part_number_elem = item.find("ns:PartNumber", ns)
    part_term_elem = item.find("ns:PartTerminologyID", ns)
    part_number = part_number_elem.text if part_number_elem is not None else None
    part_terminology_id = part_term_elem.text if part_term_elem is not None else None

    # Extract ExtendedInformation values
    for ext in item.findall("ns:ExtendedInformation/ns:ExtendedProductInformation", ns):
        extended_info_data.append({
            "PartNumber": part_number,
            "PartTerminologyID": part_terminology_id,
            "EXPICode": ext.attrib.get("EXPICode"),
            "Value": ext.text
        })

    # Extract ProductAttributes values
    for attr in item.findall("ns:ProductAttributes/ns:ProductAttribute", ns):
        product_attr_data.append({
            "PartNumber": part_number,
            "PartTerminologyID": part_terminology_id,
            "AttributeID": attr.attrib.get("AttributeID"),
            "AttributeUOM": attr.attrib.get("AttributeUOM", ""),
            "Value": attr.text
        })

# Create DataFrames for the extracted data
df_extended = pd.DataFrame(extended_info_data)
df_product_attr = pd.DataFrame(product_attr_data)

df_product_attr

,PartNumber,PartTerminologyID,AttributeID,AttributeUOM,Value
0,CP4114,2208,California Proposition 65,,WARNING: Cancer and Reproductive Harm -- www.P...
1,CP4114,2208,354,,No
2,CP4114,2208,357,,Aluminum
3,CP4114,2208,360,MM,107.2
4,CP4114,2208,360,MM,107.20
...,...,...,...,...,...
9555,CP6943,11599,3700,MM,20
9556,CP6943,11599,230,,No
9557,CP6943,11599,501,,Clockwise (Right)
9558,CP6943,11599,11118,,No


In [15]:
import pandas as pd

# DataFrames with common 'Name' column
df1 = pd.DataFrame({'Name': ['John', 'Mary', 'Bob'], 'Age': [28, 25, 30]})
df2 = pd.DataFrame({'Name': ['Mary', 'Bob', 'John'], 'City': ['Chicago', 'LA', 'NY']})

# Use map to add the 'City' to df1
# We set the index of df2 to 'Name' for the mapping
df1['City'] = df1['Name'].map(df2.set_index('Name')['City'])
df1


,Name,Age,City
0,John,28,NY
1,Mary,25,Chicago
2,Bob,30,LA


In [16]:
df_product_attr["PartName"]=df_product_attr["PartTerminologyID"].map(
    df_map[["PartTerminologyID","PartTerminologyName"]].drop_duplicates()
    .set_index(["PartTerminologyID"])["PartTerminologyName"]
    )

df_product_attr["Attribute Name"]=df_product_attr["AttributeID"].map(
    df_map[["PAID","PAName"]].drop_duplicates()
    .set_index(["PAID"])["PAName"]
    )

df_product_attr['Attribute Name']=np.where(df_product_attr['Attribute Name'].isna(), df_product_attr['AttributeID'], df_product_attr['Attribute Name'])

df_product_attr=df_product_attr[["PartNumber","PartName","Attribute Name","Value","AttributeUOM"]]

In [17]:
df_product_attr

,PartNumber,PartName,Attribute Name,Value,AttributeUOM
0,CP4114,Engine Water Pump,California Proposition 65,WARNING: Cancer and Reproductive Harm -- www.P...,
1,CP4114,Engine Water Pump,Fan Clutch Included,No,
2,CP4114,Engine Water Pump,Housing Material,Aluminum,
3,CP4114,Engine Water Pump,Hub Height,107.2,MM
4,CP4114,Engine Water Pump,Hub Height,107.20,MM
...,...,...,...,...,...
9555,CP6943,Engine Auxiliary Water Pump,Outlet Pipe Diameter,20,MM
9556,CP6943,Engine Auxiliary Water Pump,Pulley Included,No,
9557,CP6943,Engine Auxiliary Water Pump,Rotation,Clockwise (Right),
9558,CP6943,Engine Auxiliary Water Pump,Shaft Included,No,


In [18]:
len(df_product_attr['PartNumber'].unique().tolist())

465

In [19]:
# Write the DataFrames to an Excel file with separate sheets
output_file = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\OReilly\Input\Updates\PIES_FULL_GQXS_Murray Temperature Control_20260219_190932.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    #df_extended.to_excel(writer, sheet_name="ExtendedInformation", index=False)
    df_product_attr.to_excel(writer, sheet_name="ProductAttributes", index=False)

print(f"Data has been extracted and saved to {output_file}")


Data has been extracted and saved to C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\OReilly\Input\Updates\PIES_FULL_GQXS_Murray Temperature Control_20260219_190932.xlsx


# ACES Files

## Batch File Extraction

In [38]:
import os

path = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat"
xml_files = []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".xml"):
            xml_files.append(os.path.join(root, file))

print("XML files found:")
for file in xml_files:
    print(file)


XML files found:
C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Aero Rotor Kit\Centric_StopTech Aero Rotor Kits ACES_2025-11-10_FULL.xml
C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Aero Rotors\Centric_StopTech Aero Rotors ACES_2025-11-10_FULL.xml
C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Centric Premium Brake Rotors\Centric_Premium Brake Rotors ACES_2025-11-10_FULL.xml
C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Cryo Brake Rotor\Centric_StopTech Cryo Brake Rotors ACES_2025-11-10_FULL.xml
C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\CTEK Brake Rotors\Centric_C-Tek Brake Rotors ACES_2025-11-10_FULL.xml
C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\FMP Rotors\Centric_FactoryMotorParts_Painted Rotors ACES_2025-11-10_

In [40]:
for xml_file in xml_files:
    print(f"Processing file: {file}")
    # Add your XML processing code here
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # ---- Extract Header Information ----
    header = root.find("Header")
    header_data = {child.tag: child.text for child in header if child.text}

    # Approved countries (multiple <Country> tags)
    countries = [c.text for c in header.findall("ApprovedFor/Country")]
    header_data["ApprovedCountries"] = ", ".join(countries)

    df_header = pd.DataFrame([header_data])
    # ---- Extract Applications ----
    apps = []
    for app in root.findall("App"):
        app_data = {
            # "AppID": app.attrib.get("id"),
            # "Action": app.attrib.get("action"),
            # "Validate": app.attrib.get("validate"),
            "BaseVehicle": app.findtext("BaseVehicle"),
            "BaseVehicleID": app.find("BaseVehicle").attrib.get("id"),
            "Qty": app.findtext("Qty"),
            "PartType": app.findtext("PartType"),
            # "PartTypeID": app.find("PartType").attrib.get("id"),
            "MfrLabel": app.findtext("MfrLabel"),
            "Position": app.findtext("Position"),
            # "PositionID": app.find("Position").attrib.get("id"),
            "Part": app.findtext("Part"),
            # "Aspiration":app.findtext("Aspiration"),
            # "EngineType":app.findtext("EngineType"),
            "MfrLabel":app.findtext("MfrLabel"),
            # "Note":app.findtext("Note"),
            # "CylinderHeadType":app.findtext("CylinderHeadType"),
            # "ValvesPerEngine":app.findtext("ValvesPerEngine"),
            # "EngineBase":app.findtext("EngineBase"),
            # "BodyType": app.findtext("BodyType"),
            # "DriveType": app.findtext("DriveType"),
            # "FuelType": app.findtext("FuelType"),
            # "Region": app.findtext("Region"),
            # "SubModel": app.findtext("SubModel"),
            # "BodyNumDoors": app.findtext("BodyNumDoors"),
            # "MfrBodyCode": app.findtext("MfrBodyCode"),
            # "FrontBrakeType": app.findtext("FrontBrakeType"),
            # "RearBrakeType": app.findtext("RearBrakeType"),
            # "BrakeSystem": app.findtext("BrakeSystem"),
            # "BrakeABS": app.findtext("BrakeABS"),
            # "FrontSpringType": app.findtext("FrontSpringType"),
            # "RearSpringType": app.findtext("RearSpringType"),
            # "SteeringType": app.findtext("SteeringType"),
            # "EngineDesignation": app.findtext("EngineDesignation"),
            
        }
        apps.append(app_data)
    df_apps = pd.DataFrame(apps)
    # df_apps[['Make', 'Model', 'Year']] = df_apps['BaseVehicle'].str.split(' - ', expand=True)
    df_apps[['Year', 'Make', 'Model']] = df_apps['BaseVehicle'].str.split(', ', expand=True)
    df_apps['Key']=df_apps['Part']+"$"+df_apps['PartType']+"$"+df_apps['Year']+"$"+df_apps['Make']+"$"+df_apps['Model']+"$"+df_apps['Position']
    # df_apps["EngineLiters"]=df_apps["EngineBase"].str.extract(r'([\d\.]+)L')
    # df_apps["EngineCC"]=df_apps["EngineBase"].str.extract(r'\((\d+)\)')
    # df_apps["EngineCylinder"]=df_apps["EngineBase"].str.extract(r'(\d+)Cyl')
    # df_apps["EngineBlock"]=df_apps["EngineBase"].str.extract(r'([A-Za-z]+) \(')
    # df_apps["EngBoreMetric"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s*\(Bore\)')
    # df_apps["EngBoreInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Bore\)')
    # df_apps["EngStrokeInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Stroke\)')
    # df_apps["EngStrokeMetric"]=df_apps["EngineBase"].str.extract(r'\d+\.\d+\s+(\d+(?:\.\d+)?)\s*\(Stroke\)')
    del df_apps["BaseVehicle"]
    output_file=rf"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\CSV_Files\{xml_file.split("\\")[-2]}_PartCat_Apps.csv"
    df_apps.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Data has been extracted and saved to {output_file}")

Processing file: C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Sport Cryo Brake FRotor\Centric_StopTech Sport Cryo Brake Rotors ACES_2025-11-10_FULL.xml
Data has been extracted and saved to C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\CSV_Files\Aero Rotor Kit_PartCat_Apps.csv
Processing file: C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Sport Cryo Brake FRotor\Centric_StopTech Sport Cryo Brake Rotors ACES_2025-11-10_FULL.xml
Data has been extracted and saved to C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\CSV_Files\Aero Rotors_PartCat_Apps.csv
Processing file: C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\PartCat\Sport Cryo Brake FRotor\Centric_StopTech Sport Cryo Brake Rotors ACES_2025-11-10_FULL.xml
Data has been extracted and saved to C:\Users\vikram.vadhirajan\OneDr

In [28]:
df_apps

,BaseVehicleID,Qty,PartType,MfrLabel,Position,Part,Make,Model,Year
0,5053,2,Disc Brake Rotor and Hub Assembly,Centric Premium,Front,120.65001,Ford,E-150 Econoline,1980
1,5088,2,Disc Brake Rotor and Hub Assembly,Centric Premium,Front,120.65001,Ford,E-150 Econoline Club Wagon,1980
2,5100,2,Disc Brake Rotor and Hub Assembly,Centric Premium,Front,120.65001,Ford,E-150 Econoline Club Wagon,1983
3,5082,2,Disc Brake Rotor and Hub Assembly,Centric Premium,Front,120.65001,Ford,E-150 Econoline Club Wagon,1985
4,5104,2,Disc Brake Rotor and Hub Assembly,Centric Premium,Front,120.65001,Ford,E-150 Econoline Club Wagon,1990
...,...,...,...,...,...,...,...,...,...
95836,162710,2,Disc Brake Rotor,Centric Premium,Rear,120.67091,Ram,ProMaster 3500,2023
95837,159010,2,Disc Brake Rotor,Centric Premium,Rear,120.67091,Ram,ProMaster 3500,2022
95838,173116,2,Disc Brake Rotor,Centric Premium,Rear,120.67091,Ram,ProMaster 1500,2024
95839,173120,2,Disc Brake Rotor,Centric Premium,Rear,120.67091,Ram,ProMaster 2500,2024


## Single File Extraction

In [6]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file (update the filename if needed)
xml_file = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Comparison_File_rotor\Rotor & Brake Rotor Hub Assembly compare files\CTEK Brake Rotors\First Brands Group - Brake Rotors_20251111_045439.xml"

tree = ET.parse(xml_file)
root = tree.getroot()

# ---- Extract Header Information ----
header = root.find("Header")
header_data = {child.tag: child.text for child in header if child.text}

# Approved countries (multiple <Country> tags)
countries = [c.text for c in header.findall("ApprovedFor/Country")]
header_data["ApprovedCountries"] = ", ".join(countries)

df_header = pd.DataFrame([header_data])


In [ ]:

# ---- Extract Applications ----
apps = []
for app in root.findall("App"):
    app_data = {
        "AppID": app.attrib.get("id"),
        "Action": app.attrib.get("action"),
        "Validate": app.attrib.get("validate"),
        "BaseVehicle": app.findtext("BaseVehicle"),
        "BaseVehicleID": app.find("BaseVehicle").attrib.get("id"),
        "Qty": app.findtext("Qty"),
        "PartType": app.findtext("PartType"),
        "PartTypeID": app.find("PartType").attrib.get("id"),
        "MfrLabel": app.findtext("MfrLabel"),
        "Position": app.findtext("Position"),
        "PositionID": app.find("Position").attrib.get("id"),
        "Part": app.findtext("Part"),
        "Aspiration":app.findtext("Aspiration"),
        "EngineType":app.findtext("EngineType"),
        "MfrLabel":app.findtext("MfrLabel"),
        "Note":app.findtext("Note"),
        "CylinderHeadType":app.findtext("CylinderHeadType"),
        "ValvesPerEngine":app.findtext("ValvesPerEngine"),
        "EngineBase":app.findtext("EngineBase"),
        "BodyType": app.findtext("BodyType"),
        "DriveType": app.findtext("DriveType"),
        "FuelType": app.findtext("FuelType"),
        "Region": app.findtext("Region"),
        "SubModel": app.findtext("SubModel"),
        "BodyNumDoors": app.findtext("BodyNumDoors"),
        "MfrBodyCode": app.findtext("MfrBodyCode"),
        "FrontBrakeType": app.findtext("FrontBrakeType"),
        "RearBrakeType": app.findtext("RearBrakeType"),
        "BrakeSystem": app.findtext("BrakeSystem"),
        "BrakeABS": app.findtext("BrakeABS"),
        "FrontSpringType": app.findtext("FrontSpringType"),
        "RearSpringType": app.findtext("RearSpringType"),
        "SteeringType": app.findtext("SteeringType"),
        "EngineDesignation": app.findtext("EngineDesignation"),
        
    }
    apps.append(app_data)
df_apps = pd.DataFrame(apps)


In [ ]:
df_apps[['Year', 'Make', 'Model']] = df_apps['BaseVehicle'].str.split(', ', expand=True)

In [12]:
df_apps

,AppID,Action,Validate,BaseVehicle,BaseVehicleID,Qty,PartType,PartTypeID,MfrLabel,Position,...,RearBrakeType,BrakeSystem,BrakeABS,FrontSpringType,RearSpringType,SteeringType,EngineDesignation,Year,Make,Model
0,1,A,yes,Jeep - J10 - 1981,2270,2,Disc Brake Rotor,1896,Centric CTEK,Front,...,None,None,None,None,None,None,None,Jeep,J10,1981
1,2,A,yes,Jeep - J10 - 1984,2273,2,Disc Brake Rotor,1896,Centric CTEK,Front,...,None,None,None,None,None,None,None,Jeep,J10,1984
2,3,A,yes,Jeep - Cherokee - 1982,2217,2,Disc Brake Rotor,1896,Centric CTEK,Front,...,None,None,None,None,None,None,None,Jeep,Cherokee,1982
3,4,A,yes,Jeep - Wagoneer - 1982,2306,2,Disc Brake Rotor,1896,Centric CTEK,Front,...,None,None,None,None,None,None,None,Jeep,Wagoneer,1982
4,5,A,yes,Jeep - Wagoneer - 1981,2305,2,Disc Brake Rotor,1896,Centric CTEK,Front,...,None,None,None,None,None,None,None,Jeep,Wagoneer,1981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79143,79144,A,yes,Mercedes-Benz - GLE63 AMG S - 2024,173183,2,Disc Brake Rotor,1896,Centric CTEK,Rear,...,None,None,None,None,None,None,None,Mercedes-Benz,GLE63 AMG S,2024
79144,79145,A,yes,Mercedes-Benz - GLS450 - 2025,179016,2,Disc Brake Rotor,1896,Centric CTEK,Rear,...,None,None,None,None,None,None,None,Mercedes-Benz,GLS450,2025
79145,79146,A,yes,Chevrolet - Express 3500 - 1997,3314,2,Disc Brake Rotor and Hub Assembly,10292,Centric CTEK,Front,...,None,None,None,None,None,None,None,Chevrolet,Express 3500,1997
79146,79147,A,yes,Chevrolet - Express 3500 - 1997,3314,2,Disc Brake Rotor and Hub Assembly,10292,Centric CTEK,Front,...,None,None,None,None,None,None,None,Chevrolet,Express 3500,1997


In [5]:
df_apps["EngineLiters"]=df_apps["EngineBase"].str.extract(r'([\d\.]+)L')
df_apps["EngineCC"]=df_apps["EngineBase"].str.extract(r'\((\d+)\)')
df_apps["EngineCylinder"]=df_apps["EngineBase"].str.extract(r'(\d+)Cyl')
df_apps["EngineBlock"]=df_apps["EngineBase"].str.extract(r'([A-Za-z]+) \(')
df_apps["EngBoreMetric"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s*\(Bore\)')
df_apps["EngBoreInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Bore\)')
df_apps["EngStrokeInch"]=df_apps["EngineBase"].str.extract(r'(\d+\.\d+)\s+\d+(?:\.\d+)?\s*\(Stroke\)')
df_apps["EngStrokeMetric"]=df_apps["EngineBase"].str.extract(r'\d+\.\d+\s+(\d+(?:\.\d+)?)\s*\(Stroke\)')

In [73]:
df_Filtered=pd.merge(df_apps,df_map[['Article Number','Parent article number']],left_on='Part',right_on='Article Number',how='inner')
df_Filtered

,AppID,Action,Validate,BaseVehicle,BaseVehicleID,Qty,PartType,PartTypeID,MfrLabel,Position,...,EngineLiters,EngineCC,EngineCylinder,EngineBlock,EngBoreMetric,EngBoreInch,EngStrokeInch,EngStrokeMetric,Article Number,Parent article number
0,102,A,yes,"1995, Suzuki, Esteem",3,2,Disc Brake Rotor,1896,CTEK Standard Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.48007,86611
1,132,A,yes,"1996, Suzuki, Esteem",4,2,Disc Brake Rotor,1896,CTEK Standard Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.48007,86611
2,162,A,yes,"1997, Suzuki, Esteem",5,2,Disc Brake Rotor,1896,CTEK Standard Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.48007,86611
3,192,A,yes,"1998, Suzuki, Esteem",6,2,Disc Brake Rotor,1896,CTEK Standard Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.48007,86611
4,262,A,yes,"1999, Suzuki, Esteem",7,2,Disc Brake Rotor,1896,CTEK Standard Disc Brake Rotors,Front,...,1.6,1590,4,L,75.2,2.96,3.54,90.0,121.48007,86611
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76714,2782289,A,yes,"2023, Ford, E-350 Super Duty",160511,2,Disc Brake Rotor and Hub Assembly,10292,CTEK Standard Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.65124,880681
76715,2782308,A,yes,"2023, Ford, E-350 Super Duty",160511,2,Disc Brake Rotor and Hub Assembly,10292,Centric Premium Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,120.65124,880681
76716,2782329,A,yes,"2023, Ford, E-350 Super Duty",160511,2,Disc Brake Rotor and Hub Assembly,10292,Centric Premium Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,120.65126,880626
76717,2782349,A,yes,"2023, Ford, E-450 Super Duty",162664,2,Disc Brake Rotor and Hub Assembly,10292,CTEK Standard Disc Brake Rotors,Front,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.65126,880626


In [ ]:
#df_Filtered=df_Filtered[['Parent article number','Article Number','PartType',  'Part', 'Year','Make', 'Model', 'Position']]

In [74]:
# ---- Save to Excel ----
output_file = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Catalog\Catalog System Consolidation\Project Unify Files\Project Unify 3.0\Centric Rotor Catalog Data Validation\Centric\Centric_ACES.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_Filtered.to_excel(writer, sheet_name="ACES", index=False)
print(f"Data has been extracted and saved to {output_file}")


Data has been extracted and saved to C:\Users\vikram.vadhirajan\OneDrive - Trico\Catalog\Catalog System Consolidation\Project Unify Files\Project Unify 3.0\Centric Rotor Catalog Data Validation\Centric\Centric_ACES.xlsx


# Misc Code

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

# Load the XML file
xml_file = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Catalog\Catalog System Consolidation\Project Unify Files\Project Unify 3.0\Centric Rotor Catalog Data Validation\Centric\Centric Parts_Centric Parts Pies File_2025-09-01.xml"
tree = ET.parse(xml_file)
root = tree.getroot()

# Define namespaces (AutoCare PIES XML uses default namespace)
ns = {"ns": "http://www.autocare.org"}

# Extract Header Information
header = root.find("ns:Header", ns)
header_data = {child.tag.split("}")[1]: child.text for child in header}
df_header = pd.DataFrame([header_data])

# Extract Price Sheets
pricesheets = []
for ps in root.findall("ns:PriceSheets/ns:PriceSheet", ns):
    ps_data = {child.tag.split("}")[1]: child.text for child in ps}
    pricesheets.append(ps_data)
df_pricesheets = pd.DataFrame(pricesheets)

# Extract Items
items = []
extended_info = []
product_attributes = []

for item in root.findall("ns:Items/ns:Item", ns):
    part_number = item.find("ns:PartNumber", ns).text
    # brand_label = item.find("ns:BrandLabel", ns).text

    # Extract Item Data
    item_data = {
        "PartNumber": part_number,
        "BrandLabel": brand_label,
        # "GTIN": item.find("ns:ItemLevelGTIN", ns).text,
        # "MinimumOrderQuantity": item.find("ns:MinimumOrderQuantity", ns).text,
    }

    # Extract Descriptions
    descriptions = item.findall("ns:Descriptions/ns:Description", ns)
    for desc in descriptions:
        desc_code = desc.attrib.get("DescriptionCode", "Other")
        item_data[f"Description_{desc_code}"] = desc.text

    # Extract Prices
    prices = item.findall("ns:Prices/ns:Pricing", ns)
    for price in prices:
        price_type = price.attrib.get("PriceType", "Other")
        item_data[f"Price_{price_type}"] = price.find("ns:Price", ns).text

    items.append(item_data)

    # Extract Extended Information
    for ext in item.findall("ns:ExtendedInformation/ns:ExtendedProductInformation", ns):
        extended_info.append({
            "PartNumber": part_number,
            "EXPICode": ext.attrib.get("EXPICode"),
            "Value": ext.text
        })

    # Extract Product Attributes
    for attr in item.findall("ns:ProductAttributes/ns:ProductAttribute", ns):
        product_attributes.append({
            "PartNumber": part_number,
            "AttributeID": attr.attrib.get("AttributeID"),
            "Value": attr.text
        })

df_items = pd.DataFrame(items)
df_extended_info = pd.DataFrame(extended_info)
df_product_attributes = pd.DataFrame(product_attributes)

# Extract Marketing Copy
marketing_copy = []
for mc in root.findall("ns:MarketingCopy/ns:MarketCopy/ns:MarketCopyContent", ns):
    mc_data = {
        "MarketCopyCode": mc.attrib.get("MarketCopyCode"),
        "MarketCopyType": mc.attrib.get("MarketCopyType"),
        "Content": mc.text,
    }
    marketing_copy.append(mc_data)

df_marketing_copy = pd.DataFrame(marketing_copy)



# Save to Excel with different sheets
output_file = r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Import files\Rest\Lumileds.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_header.to_excel(writer, sheet_name="Header", index=False)
    df_pricesheets.to_excel(writer, sheet_name="PriceSheets", index=False)
    df_items.to_excel(writer, sheet_name="Items", index=False)
    df_extended_info.to_excel(writer, sheet_name="ExtendedInformation", index=False)
    df_product_attributes.to_excel(writer, sheet_name="ProductAttributes", index=False)
    df_marketing_copy.to_excel(writer, sheet_name="MarketingCopy", index=False)

print(f"Data has been extracted and saved to {output_file}")

